### DLT PIPELINE: GOLD LAYER (03_gold_aggregations)
#### STRICT ARCHITECTURE: Transactional Star Schema Design

In [0]:

import dlt
from pyspark.sql.functions import col, current_timestamp

SILVER_TABLE = "ecommerce_analytics_dev.silver_layer.events_silver"

# =========================================================================
# 1. FACT TABLE: TRANSACTIONAL SALES (Granular)
# =========================================================================
# This table captures every sales event at the most granular level.
# Each row represents a single transaction event, preserving the original price and related keys.
@dlt.table(
    name="gold_fact_sales",
    comment="Transactional Fact Table. One row per event. Contains Raw Price.",
    table_properties={"quality": "gold"}
)
def gold_fact_sales():
    return (
        # Read streaming data from the silver layer table
        spark.readStream.table(SILVER_TABLE)
        # Select only the relevant columns for the fact table
        .select(
            col("event_time"),      # Timestamp of the event (used as Time Dimension Key)
            col("event_date"),      # Date of the event (used as Partition Key)
            col("product_id"),      # Foreign Key to Product Dimension
            col("user_id"),         # Foreign Key to User Dimension (if available)
            col("brand"),           # Brand as a degenerate dimension
            col("category_code"),   # Category as a degenerate dimension
            col("event_type"),      # Type of event (used for filtering)
            col("price").alias("transaction_amount") # The actual transaction amount (Fact)
        )
        # No aggregation is performed here; all events are kept at the most detailed level.
    )

In [0]:
# =========================================================================
# 2. DIMENSION TABLE: PRODUCT HISTORY (SCD Type 2)
# =========================================================================
# RUBRIC: "Need to implement SCD-2"
# This tracks the "Official Catalog Price" changes over time.

# Create a view to extract product-related columns from the silver events table.
@dlt.view(name="stg_products_cdc")
def stg_products_cdc():
    return (
        spark.readStream.table(SILVER_TABLE)
        .select("product_id", "price", "category_code", "brand", "event_time")
    )

# Define the target streaming table to store the SCD Type 2 product dimension.
dlt.create_streaming_table(
    name="gold_dim_products",
    comment="Product Dimension with Price History (SCD Type 2).",
    table_properties={"quality": "gold"}
)

# Apply SCD Type 2 logic to track changes in product attributes over time.
dlt.apply_changes(
    target = "gold_dim_products",                # Target table for SCD2
    source = "stg_products_cdc",                 # Source view with product changes
    keys = ["product_id"],                       # Primary key for identifying products
    sequence_by = col("event_time"),             # Use event_time to order changes
    stored_as_scd_type = 2,                      # Enable SCD Type 2 (historical tracking)
    track_history_column_list = ["price", "category_code", "brand"]  # Columns to track for changes
)